interesting traits -> >= 1 genetic association

In [1]:
import pandas as pd
import os
import glob
import pickle

In [2]:
data_dir = "../data/"
files = glob.glob(f"{data_dir+"association_by_datasource_direct"}/*.parquet")

In [3]:

genetic_association_sources = [
    'genomics_england','orphanet','gwas_credible_sets','eva',
    'uniprot_literature', 'gene_burden', 'uniprot_variants',
    'gene2phenotype', 'clingen'
]

score_thresholds = {
    'genomics_england':0.5,
    'orphanet': 1,
    'gwas_credible_sets':0.5,
    'eva': 0.5,
    'uniprot_literature': 0.5, 
    'gene_burden': 0.25, 
    'uniprot_variants': 0.5,
    'gene2phenotype': 0.5, 
    'clingen': 0.5 
}

In [4]:
chunk_size = 5
filtered_chunks = []
#assoc_chunk = pd.concat([pd.read_parquet(f) for f in files[indexes[0]:indexes[-1]+1]], ignore_index=True)


In [5]:
for i in range(0, len(files), chunk_size):
    print(f"Reading and filtering chunk: {i}")
    chunk_files = files[i:i + chunk_size]  # slicing handles the end boundary cleanly
    assoc_chunk = pd.concat([pd.read_parquet(f) for f in chunk_files], ignore_index=True)
    assoc_chunk = assoc_chunk[assoc_chunk['aggregationValue'].isin(genetic_association_sources)].reset_index(drop=True)
    assoc_chunk['threshold'] = assoc_chunk.apply(lambda x: score_thresholds[x['aggregationValue']], axis=1)
    assoc_chunk = assoc_chunk[assoc_chunk['associationScore'] >= assoc_chunk['threshold']].reset_index(drop=True)
    filtered_chunks.append(assoc_chunk.copy())  # no need to wrap in pd.DataFrame()

Reading and filtering chunk: 0
Reading and filtering chunk: 5
Reading and filtering chunk: 10
Reading and filtering chunk: 15
Reading and filtering chunk: 20
Reading and filtering chunk: 25
Reading and filtering chunk: 30
Reading and filtering chunk: 35
Reading and filtering chunk: 40
Reading and filtering chunk: 45
Reading and filtering chunk: 50


In [6]:
genetic_associations =  pd.concat(filtered_chunks)

In [7]:
genetic_associations.shape

(250578, 9)

In [8]:
traits = list(genetic_associations['diseaseId'].unique())

In [9]:
with open(os.path.join("..","aux_data","traits_geq_oneGenAssoc.pkl"),"wb") as file:
    pickle.dump(traits, file)

In [11]:
diseases = pd.read_parquet("../data/disease/disease.parquet")
diseases.head(1)

,id,code,name,description,dbXRefs,parents,exactSynonyms,relatedSynonyms,narrowSynonyms,broadSynonyms,obsoleteTerms,obsoleteXRefs,children,ancestors,therapeuticAreas,descendants,ontology,synonyms
0,DOID_0050890,http://purl.obolibrary.org/obo/DOID_0050890,synucleinopathy,A neurodegenerative disease that is characteri...,"[MEDGEN:1682194, MESH:D000080874, MONDO:000051...","[EFO_0005772, MONDO_0021179]","[alpha Synucleinopathies, synucleinopathy]","[alpha synucleinopathies, synucleinopathies]",[],[],[],[],"[EFO_0006792, EFO_1001050]","[EFO_0005772, MONDO_0021179, EFO_0009386, EFO_...","[EFO_0000618, OTAR_0000020]","[EFO_0006792, EFO_1001050, MONDO_0003122, MOND...","{'isTherapeuticArea': False, 'leaf': False, 's...",{'hasExactSynonym': ['alpha Synucleinopathies'...


In [18]:
diseases[diseases['id']=='MONDO_0004975']

,id,code,name,description,dbXRefs,parents,exactSynonyms,relatedSynonyms,narrowSynonyms,broadSynonyms,obsoleteTerms,obsoleteXRefs,children,ancestors,therapeuticAreas,descendants,ontology,synonyms
4820,MONDO_0004975,http://purl.obolibrary.org/obo/MONDO_0004975,Alzheimer disease,"A progressive, neurodegenerative disease chara...","[DOID:10652, HP:0002511, ICD10CM:G30, ICD10WHO...","[MONDO_0001627, EFO_0005815]","[AD, Alzheimer dementia, Alzheimer disease, Al...",[],"[Alzheimer disease, familial]",[],[EFO_0000249],"[DOID:10652, ICD10:G30, ICD9:290.1, ICD9:331.0...","[MONDO_0014265, MONDO_0100087, EFO_0022957, EF...","[MONDO_0001627, EFO_0005815, MONDO_0002039, HP...","[MONDO_0002025, EFO_0000618, EFO_0000651]","[MONDO_0014265, MONDO_0100087, EFO_0022957, EF...","{'isTherapeuticArea': False, 'leaf': False, 's...","{'hasExactSynonym': ['AD', 'Alzheimer dementia..."


In [17]:
diseases.iloc[2]#['dbXRefs']

id                                                         DOID_10718
code                        http://purl.obolibrary.org/obo/DOID_10718
name                                                       giardiasis
description         An infection of the small intestine caused by ...
dbXRefs             [DOID:10718, ICD10CM:A07.1, ICD9:007.1, ICD9CM...
parents                                  [EFO_0009561, MONDO_0002428]
exactSynonyms       [Giardia infection, beaver feaver, beaver feve...
relatedSynonyms     [Giardia, Giardiases, Lambliases, infections, ...
narrowSynonyms                                                     []
broadSynonyms                                                      []
obsoleteTerms                                                      []
obsoleteXRefs                                                      []
children                                                           []
ancestors           [EFO_0009561, MONDO_0002428, MONDO_0043424, EF...
therapeuticAreas    